In [1]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
import requests
import time

In [2]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
df = pd.read_csv('column_looks_like_paper_filtered.csv')
print(f"Loaded {len(df)} transactions, {df['From'].nunique()} unique sender addresses")

# Keep only successful transactions
df = df[df['Status'] == 'Success'].copy()
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce').fillna(0)
df['Txn Fee'] = pd.to_numeric(df['Txn Fee'], errors='coerce').fillna(0)
df = df.sort_values('Timestamp').reset_index(drop=True)


Loaded 33385 transactions, 2993 unique sender addresses


In [3]:
# 2. FEATURE ENGINEERING 

ETHERSCAN_API_KEY = 'SKUNVYVRYG7V5IYMJJ4X9JWS6F54FZKKXH' 

def get_contract_opcodes(address):
    """
    Fetch contract bytecode from Etherscan and compute
    a simple opcode count / intensity as a numeric feature.
    """
    url = f"https://api.etherscan.io/api?module=contract&action=getsourcecode&address={address}&apikey={ETHERSCAN_API_KEY}"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        result = r.json()['result'][0]
        bytecode = result['SourceCode'] if result['SourceCode'] else result['ABI']
        # Simple proxy for opcode complexity: count length or number of instructions
        if bytecode and len(bytecode) > 0:
            opcode_count = len(bytecode)
        else:
            opcode_count = 0
        time.sleep(0.2)  # rate limit
        return opcode_count
    except:
        return 0

def extract_ttps_features(df):
    records = []

    all_addresses = set(df['From'].tolist()) | set(df['To'].tolist())

    for addr in all_addresses:
        sent = df[df['From'] == addr]
        recv = df[df['To'] == addr]

        if len(sent) == 0 and len(recv) == 0:
            continue

        # ── TTPS features ─────────────────────────
        NI = len(recv)
        NP = len(sent)

        partners = set(sent['To'].tolist()) | set(recv['From'].tolist())
        p = len(partners)
        if p >= 3:
            V = []
            for partner in partners:
                mi = len(df[(df['From'] == addr) & (df['To'] == partner)])
                ni_p = len(df[(df['From'] == partner) & (df['To'] == addr)])
                V.append(ni_p - mi)
            V = np.array(V, dtype=float)
            std = V.std()
            D_ind = float(pd.Series(V).skew()) if std > 0 else 0.0
        else:
            D_ind = 0.0

        if len(recv) > 0:
            total_recv_amount = recv['Amount'].sum()
            if total_recv_amount > 0:
                recv_sorted = recv.sort_values('Timestamp').reset_index(drop=True)
                n = len(recv_sorted)
                R_PT = float(sum(recv_sorted.loc[i, 'Amount']*((i+1)/n) for i in range(n))/total_recv_amount)
            else:
                R_PT = 0.5
        else:
            R_PT = 0.5

        T = len(sent)
        if T > 0 and len(sent['To']) > 0:
            R_mn = sent['To'].value_counts().iloc[0]
            R_MN = R_mn / T
        else:
            R_MN = 0.0

        P_total = sent['Amount'].sum()
        if P_total > 0 and len(sent) > 0:
            P_mn = sent.groupby('To')['Amount'].sum().max()
            R_MM = float(P_mn / P_total)
        else:
            R_MM = 0.0

        total_in = recv['Amount'].sum()
        total_out = sent['Amount'].sum()
        balance = total_in - total_out
        R_Bal = float(balance / total_in) if total_in > 0 else 0.0

        avg_fee = sent['Txn Fee'].mean() if len(sent) > 0 else 0.0
        tx_count = len(sent) + len(recv)

        # ── OPCODE FEATURE ───────────────────────
        opcode_feature = get_contract_opcodes(addr)

        records.append({
            'address': addr,
            'NI': NI,
            'NP': NP,
            'D_ind': D_ind,
            'R_PT': R_PT,
            'R_MN': R_MN,
            'R_MM': R_MM,
            'R_Bal': R_Bal,
            'avg_fee': avg_fee,
            'tx_count': tx_count,
            'opcode_count': opcode_feature  # NEW FEATURE
        })

    return pd.DataFrame(records)

features_df = extract_ttps_features(df)
print(f"\nExtracted features for {len(features_df)} addresses")
print(features_df.describe().round(4))



Extracted features for 5731 addresses
              NI         NP      D_ind       R_PT       R_MN       R_MM  \
count  5731.0000  5731.0000  5731.0000  5731.0000  5731.0000  5731.0000   
mean      5.7793     5.7793    -0.0006     0.5215     0.5175     0.5180   
std       9.8205     5.9261     0.0707     0.0241     0.4996     0.4996   
min       0.0000     0.0000    -4.9942     0.1416     0.0000     0.0000   
25%       0.0000     0.0000     0.0000     0.5000     0.0000     0.0000   
50%       0.0000    11.0000     0.0000     0.5000     1.0000     1.0000   
75%      11.0000    11.0000     0.0000     0.5455     1.0000     1.0000   
max     561.0000   110.0000     1.7321     0.7605     1.0000     1.0000   

             R_Bal    avg_fee   tx_count  opcode_count  
count    5731.0000  5731.0000  5731.0000        5731.0  
mean      -52.0847     0.0000    11.5585           0.0  
std      2196.0448     0.0000     8.8729           0.0  
min   -124999.0000     0.0000    11.0000           0.0  


In [4]:
# 3. ADASYN IMPLEMENTATION 

def adasyn_oversample(X_minority, X_majority, beta=1.0, K=5):
    from sklearn.neighbors import NearestNeighbors
    n_min = len(X_minority)
    n_maj = len(X_majority)
    G = int((n_maj - n_min) * beta)
    if G == 0 or n_min == 0:
        return X_minority
    X_all = np.vstack([X_minority, X_majority])
    y_all = np.array([1]*n_min + [0]*n_maj)
    knn = NearestNeighbors(n_neighbors=min(K+1, len(X_all)))
    knn.fit(X_all)
    _, indices = knn.kneighbors(X_minority)
    r = np.array([np.sum(y_all[indices[i][1:]]==0)/K for i in range(n_min)])
    r_sum = r.sum()
    r_hat = r/r_sum if r_sum > 0 else np.ones(n_min)/n_min
    g = np.round(r_hat*G).astype(int)
    knn_min = NearestNeighbors(n_neighbors=min(K+1, n_min))
    knn_min.fit(X_minority)
    _, min_indices = knn_min.kneighbors(X_minority)
    synthetic=[]
    for i,gi in enumerate(g):
        for _ in range(gi):
            neighbor_idx = min_indices[i][np.random.randint(1,len(min_indices[i]))]
            xa = X_minority[neighbor_idx]
            xi = X_minority[i]
            lam = np.random.rand()
            synthetic.append(xi + lam*(xa-xi))
    return np.vstack([X_minority]+[np.array(synthetic)]) if synthetic else X_minority


In [5]:
# 4. NORMALISE FEATURES (add opcode_count)
# ─────────────────────────────────────────────
feature_cols = ['NI','NP','D_ind','R_PT','R_MN','R_MM','R_Bal','avg_fee','tx_count','opcode_count']
X = features_df[feature_cols].fillna(0).values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


In [6]:
# ─────────────────────────────────────────────
# 5. ISOLATION FOREST ANOMALY DETECTION
# ─────────────────────────────────────────────
iso = IsolationForest(n_estimators=200,contamination=0.002, random_state=42)
iso.fit(X_scaled)
scores = iso.decision_function(X_scaled)
predictions = iso.predict(X_scaled)
features_df['anomaly_score'] = -scores
features_df['is_suspicious'] = (predictions==-1).astype(int)
print(f"\nSuspicious addresses flagged: {features_df['is_suspicious'].sum()} / {len(features_df)}")



Suspicious addresses flagged: 7 / 5731


In [7]:
# ── Summary table (ONLY predicted suspicious) ─────────────────────
#summary.head(20)
print("\n── Predicted Suspicious Addresses ─────────────────────────────")

summary = features_df[features_df['is_suspicious'] == 1][
    ['address','anomaly_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='anomaly_score', ascending=False).reset_index(drop=True)

summary.index += 1
print(summary.to_string())


── Predicted Suspicious Addresses ─────────────────────────────
                                      address  anomaly_score   NI   NP      R_PT      R_MN      R_MM          R_Bal
1  0x28c6c06298d514db089934071355e5743bf21d60       0.039842   99   66  0.631153  0.166667  0.959269       0.900548
2  0x45300136662dd4e58fc0df61e6290dffd992b785       0.032769  561   22  0.500891  0.500000  0.930233     -41.156863
3  0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f       0.011987  132  110  0.692649  0.100000  0.670196       0.897104
4  0x1111111254eeb25477b68fb85ed929f73a960582       0.009298   44    0  0.141633  0.000000  0.000000       1.000000
5  0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2       0.003096  110    0  0.410125  0.000000  0.000000       1.000000
6  0xbea30d78a6281b81c2b8bdfa13761cc80f8c4e1f       0.001781    0   11  0.500000  1.000000  1.000000       0.000000
7  0xd9df09b252db30f7881ff5cad96b7efabf6fe745       0.001419   11   11  0.545455  1.000000  1.000000 -124999.000000


In [8]:
df3 = pd.read_csv('column_looks_like_paper_filtered.csv')
df4=pd.read_csv("all_labeled_data_final.csv")

In [9]:
same_values = set(summary['address']).intersection(set(df4['data']))
print(same_values)

{'0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f', '0x1111111254eeb25477b68fb85ed929f73a960582', '0x45300136662dd4e58fc0df61e6290dffd992b785', '0x28c6c06298d514db089934071355e5743bf21d60', '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2'}


In [10]:
# ─────────────────────────────────────────────
# 4. NORMALISE FEATURES (add opcode_count)
# ─────────────────────────────────────────────
feature_cols = ['NI','NP','D_ind','R_PT','R_MN','R_MM','R_Bal','avg_fee','tx_count','opcode_count']
X = features_df[feature_cols].fillna(0).values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


In [29]:
# ─────────────────────────────────────────────
# 5. ISOLATION FOREST ANOMALY DETECTION 
# ─────────────────────────────────────────────
iso = IsolationForest(n_estimators=200, contamination=0.002, random_state=42)
iso.fit(X_scaled)
scores = iso.decision_function(X_scaled)
predictions = iso.predict(X_scaled)
features_df['iso'] = -scores
features_df['is_suspicious'] = (predictions==-1).astype(int)
print(f"\nSuspicious addresses flagged: {features_df['is_suspicious'].sum()} / {len(features_df)}")
# ── Summary table (ONLY predicted suspicious) ─────────────────────
#summary.head(20)
print("\n── Predicted Suspicious Addresses ─────────────────────────────")

summary = features_df[features_df['is_suspicious'] == 1][
    ['address','iso','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='iso', ascending=False).reset_index(drop=True)

summary.index += 1
print(summary.to_string())


Suspicious addresses flagged: 7 / 5731

── Predicted Suspicious Addresses ─────────────────────────────
                                      address       iso   NI   NP      R_PT      R_MN      R_MM          R_Bal
1  0x28c6c06298d514db089934071355e5743bf21d60  0.039842   99   66  0.631153  0.166667  0.959269       0.900548
2  0x45300136662dd4e58fc0df61e6290dffd992b785  0.032769  561   22  0.500891  0.500000  0.930233     -41.156863
3  0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f  0.011987  132  110  0.692649  0.100000  0.670196       0.897104
4  0x1111111254eeb25477b68fb85ed929f73a960582  0.009298   44    0  0.141633  0.000000  0.000000       1.000000
5  0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2  0.003096  110    0  0.410125  0.000000  0.000000       1.000000
6  0xbea30d78a6281b81c2b8bdfa13761cc80f8c4e1f  0.001781    0   11  0.500000  1.000000  1.000000       0.000000
7  0xd9df09b252db30f7881ff5cad96b7efabf6fe745  0.001419   11   11  0.545455  1.000000  1.000000 -124999.000000


In [41]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np

# ─────────────────────────────────────────────
# 1. ANOMALY SCORE DISTRIBUTION
# ─────────────────────────────────────────────
plt.figure(figsize=(8,5))
plt.hist(features_df['iso'], bins=50)
plt.title("Anomaly Score Distribution For Isolation Forest",fontsize=14,fontweight="bold")
plt.xlabel("Anomaly Score",fontsize=10,fontweight="bold")
plt.ylabel("Frequency",fontsize=10,fontweight="bold")
plt.xlim(-0.5,0)
plt.ylim(0,3000)
plt.savefig("anomaly_score_distribution_for_isolation_forest.png",dpi=600)
plt.close()

print("\nSaved: anomaly_score_distribution.png")

# ─────────────────────────────────────────────
# 2. TOP-K SUSPICIOUS ADDRESSES
# ─────────────────────────────────────────────
top_k = features_df.sort_values(by='iso', ascending=False).head(7)

print(f"\n── Top {7} Most Suspicious Addresses ─────────────────────────")
print(top_k[['address','iso','NI','NP','R_PT','R_MN','R_MM','R_Bal']].to_string(index=False))

# ─────────────────────────────────────────────
# 3. PCA VISUALIZATION (2D)
# ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

features_df['pca1'] = X_pca[:,0]
features_df['pca2'] = X_pca[:,1]

plt.figure(figsize=(8,6))
plt.scatter(
    features_df['pca1'],
    features_df['pca2'],
    c=features_df['is_suspicious'],
)
plt.title("PCA Projection Of Isolation Forest Detected Samples",fontsize=14,fontweight="bold")
plt.xlabel("PCA 1",fontsize=10,fontweight="bold")
plt.ylabel("PCA 2",fontsize=10,fontweight="bold")
plt.xlim(-0.8,0.8)
plt.ylim(-0.2,1.0)
plt.savefig("pca_anomaly_isolation_forest.png",dpi=600)
plt.close()

print("Saved: pca_anomaly.png")

# ─────────────────────────────────────────────
# 4. SILHOUETTE SCORE (QUALITY CHECK)
# ─────────────────────────────────────────────
# Convert anomaly labels to 0/1 clusters
labels = features_df['is_suspicious'].values

# Only compute if both classes exist
if len(np.unique(labels)) > 1:
    sil_score = silhouette_score(X_scaled, labels)
    print(f"\nSilhouette Score: {sil_score:.4f}")
else:
    print("\nSilhouette Score: Not applicable (only one class found)")


Saved: anomaly_score_distribution.png

── Top 7 Most Suspicious Addresses ─────────────────────────
                                   address      iso  NI  NP     R_PT     R_MN     R_MM          R_Bal
0x28c6c06298d514db089934071355e5743bf21d60 0.039842  99  66 0.631153 0.166667 0.959269       0.900548
0x45300136662dd4e58fc0df61e6290dffd992b785 0.032769 561  22 0.500891 0.500000 0.930233     -41.156863
0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f 0.011987 132 110 0.692649 0.100000 0.670196       0.897104
0x1111111254eeb25477b68fb85ed929f73a960582 0.009298  44   0 0.141633 0.000000 0.000000       1.000000
0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2 0.003096 110   0 0.410125 0.000000 0.000000       1.000000
0xbea30d78a6281b81c2b8bdfa13761cc80f8c4e1f 0.001781   0  11 0.500000 1.000000 1.000000       0.000000
0xd9df09b252db30f7881ff5cad96b7efabf6fe745 0.001419  11  11 0.545455 1.000000 1.000000 -124999.000000
Saved: pca_anomaly.png

Silhouette Score: 0.4414


In [14]:
# ─────────────────────────────────────────────
# 2. CONTAMINATION RATIO
# ─────────────────────────────────────────────
suspicious = features_df['is_suspicious'].sum()
total = len(features_df)

print("\n─────────────────────────────────────────")
print("Contamination Ratio")
print("─────────────────────────────────────────")
print(f"Suspicious: {suspicious}")
print(f"Total: {total}")
print(f"Ratio: {suspicious/total:.5f}")



─────────────────────────────────────────
Contamination Ratio
─────────────────────────────────────────
Suspicious: 7
Total: 5731
Ratio: 0.00122


In [15]:
same_values = set(summary['address']).intersection(set(df4['data']))
print(same_values)

{'0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f', '0x1111111254eeb25477b68fb85ed929f73a960582', '0x45300136662dd4e58fc0df61e6290dffd992b785', '0x28c6c06298d514db089934071355e5743bf21d60', '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2'}


In [16]:
# Find common addresses
common_addresses = set(df3['From']).intersection(set(df4['data']))

# Rows in summary that have matching address
matching_in_summary = summary[summary['address'].isin(common_addresses)]

# Rows in df5 that have matching values
matching_in_df4 = df4[df4['data'].isin(common_addresses)]

print("Matches found in summary:")
print(matching_in_summary)

#print("\nMatches found in df5:")
#print(matching_in_df5)

Matches found in summary:
                                      address  iso+adasyn   NI   NP      R_PT  \
1  0x28c6c06298d514db089934071355e5743bf21d60    0.039842   99   66  0.631153   
2  0x45300136662dd4e58fc0df61e6290dffd992b785    0.032769  561   22  0.500891   
3  0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f    0.011987  132  110  0.692649   

       R_MN      R_MM      R_Bal  
1  0.166667  0.959269   0.900548  
2  0.500000  0.930233 -41.156863  
3  0.100000  0.670196   0.897104  


In [17]:
# Find common addresses
common_addresses = set(df3['To']).intersection(set(df4['data']))

# Rows in summary that have matching address
matching_in_summary = summary[summary['address'].isin(common_addresses)]

# Rows in df5 that have matching values
matching_in_df4 = df4[df4['data'].isin(common_addresses)]

print("Matches found in summary:")
print(matching_in_summary)

#print("\nMatches found in df5:")
#print(matching_in_df5)

Matches found in summary:
                                      address  iso+adasyn   NI   NP      R_PT  \
1  0x28c6c06298d514db089934071355e5743bf21d60    0.039842   99   66  0.631153   
2  0x45300136662dd4e58fc0df61e6290dffd992b785    0.032769  561   22  0.500891   
3  0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f    0.011987  132  110  0.692649   
4  0x1111111254eeb25477b68fb85ed929f73a960582    0.009298   44    0  0.141633   
5  0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2    0.003096  110    0  0.410125   

       R_MN      R_MM      R_Bal  
1  0.166667  0.959269   0.900548  
2  0.500000  0.930233 -41.156863  
3  0.100000  0.670196   0.897104  
4  0.000000  0.000000   1.000000  
5  0.000000  0.000000   1.000000  


In [18]:
# ─────────────────────────────────────────────
# 6. ISOLATION FOREST + SMOTE
# ─────────────────────────────────────────────

from imblearn.over_sampling import SMOTE

# Use isolation forest labels as pseudo labels
y_smote = features_df['is_suspicious'].values

# Apply SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote_resampled = smote.fit_resample(X_scaled, y_smote)

# Train Isolation Forest again
iso_smote = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

iso_smote.fit(X_smote)

scores_smote = iso_smote.decision_function(X_scaled)
pred_smote = iso_smote.predict(X_scaled)

features_df['smote_score'] = -scores_smote
features_df['smote_flag'] = (pred_smote == -1).astype(int)

print(f"\nSMOTE Suspicious: {features_df['smote_flag'].sum()} / {len(features_df)}")

# ── SMOTE Summary ─────────────────────────────
smote_summary = features_df[features_df['smote_flag'] == 1][
    ['address','smote_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='smote_score', ascending=False).reset_index(drop=True)

smote_summary.index += 1

print("\n── Isolation Forest + SMOTE Suspicious ─────────────────────────")
print(smote_summary.to_string())

Python(28661) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



SMOTE Suspicious: 1 / 5731

── Isolation Forest + SMOTE Suspicious ─────────────────────────
                                      address  smote_score   NI  NP      R_PT  R_MN      R_MM      R_Bal
1  0x45300136662dd4e58fc0df61e6290dffd992b785     0.021806  561  22  0.500891   0.5  0.930233 -41.156863


In [40]:

# ─────────────────────────────────────────────
# 1. SMOTE ANOMALY SCORE DISTRIBUTION
# ─────────────────────────────────────────────
plt.figure(figsize=(8,5))
plt.hist(features_df['smote_score'], bins=50)
plt.title("Anomaly Score Distribution For IF+SMOTE",fontsize=14,fontweight="bold")
plt.xlabel("Anomaly Score",fontsize=10,fontweight="bold")
plt.ylabel("Frequency",fontsize=10,fontweight="bold")
plt.xlim(-0.5,0)
plt.ylim(0,3000)
plt.savefig("Anomaly_score_IF+SMOTE.png",dpi=600)
plt.close()

print("\nSaved: smote_anomaly_score_distribution.png")


# ─────────────────────────────────────────────
# 2. TOP-K SMOTE SUSPICIOUS ADDRESSES
# ─────────────────────────────────────────────
K = 7
top_k_smote = features_df.sort_values(by='smote_score', ascending=False).head(K)

print(f"\n── Top {K} SMOTE Suspicious Addresses ─────────────────────────")
print(top_k_smote[['address','smote_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']].to_string(index=False))


# ─────────────────────────────────────────────
# 3. PCA VISUALIZATION (SMOTE FLAGS)
# ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

features_df['pca1_smote'] = X_pca[:,0]
features_df['pca2_smote'] = X_pca[:,1]

plt.figure(figsize=(8,6))
plt.scatter(
    features_df['pca1_smote'],
    features_df['pca2_smote'],
    c=features_df['smote_flag'],
)
plt.title("PCA Projection Of IF+SMOTE Detected Samples ",fontsize=14,fontweight="bold")
plt.xlabel("PCA 1",fontsize=10,fontweight="bold")
plt.ylabel("PCA 2",fontsize=10,fontweight="bold")
plt.xlim(-0.8,0.8)
plt.ylim(-0.2,1.0)
plt.savefig("pca_IF+SMOTE.png",dpi=600)
plt.close()

print("Saved: pca_smote_anomaly.png")


# ─────────────────────────────────────────────
# 4. SILHOUETTE SCORE (SMOTE VERSION)
# ─────────────────────────────────────────────
labels_smote = features_df['smote_flag'].values

if len(np.unique(labels_smote)) > 1:
    sil_score_smote = silhouette_score(X_scaled, labels_smote)
    print(f"\nSMOTE Silhouette Score: {sil_score_smote:.4f}")
else:
    print("\nSMOTE Silhouette Score: Not applicable (only one class found)")


Saved: smote_anomaly_score_distribution.png

── Top 7 SMOTE Suspicious Addresses ─────────────────────────
                                   address  smote_score  NI  NP     R_PT     R_MN     R_MM          R_Bal
0x45300136662dd4e58fc0df61e6290dffd992b785     0.021806 561  22 0.500891 0.500000 0.930233     -41.156863
0x1111111254eeb25477b68fb85ed929f73a960582    -0.053862  44   0 0.141633 0.000000 0.000000       1.000000
0x4976a4a02f38326660d17bf34b431dc6e2eb2327    -0.063437   0  77 0.500000 0.142857 0.549882       0.000000
0xd9df09b252db30f7881ff5cad96b7efabf6fe745    -0.063755  11  11 0.545455 1.000000 1.000000 -124999.000000
0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23    -0.064898  44  77 0.514392 0.142857 0.909091    -188.655237
0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f    -0.068207 132 110 0.692649 0.100000 0.670196       0.897104
0x83e22b198fb9df384902e7df1aca8cd75184acdc    -0.092383  11  11 0.545455 1.000000 1.000000  -99999.000000
Saved: pca_smote_anomaly.png

SMOTE Silhouet

In [21]:
# ─────────────────────────────────────────────
# 2. CONTAMINATION RATIO
# ─────────────────────────────────────────────
suspicious = features_df['smote_flag'].sum()
total = len(features_df)

print("\n─────────────────────────────────────────")
print("Contamination Ratio")
print("─────────────────────────────────────────")
print(f"Suspicious: {suspicious}")
print(f"Total: {total}")
print(f"Ratio: {suspicious/total:.5f}")



─────────────────────────────────────────
Contamination Ratio
─────────────────────────────────────────
Suspicious: 1
Total: 5731
Ratio: 0.00017


In [22]:
same_values = set(smote_summary['address']).intersection(set(df4['data']))
print(same_values)

{'0x45300136662dd4e58fc0df61e6290dffd992b785'}


In [23]:
# ─────────────────────────────────────────────
# 7. ISOLATION FOREST + XGBOOST
# ────────────────────────────────────────────

from xgboost import XGBClassifier

# Use isolation forest labels as pseudo labels
y_xgb = features_df['is_suspicious'].values

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_scaled, y_xgb)

xgb_prob = xgb.predict_proba(X_scaled)[:,1]
xgb_pred = (xgb_prob > 0.5).astype(int)

features_df['xgb_score'] = xgb_prob
features_df['xgb_flag'] = xgb_pred

print(f"\nXGBoost Suspicious: {features_df['xgb_flag'].sum()} / {len(features_df)}")

# ── XGBoost Summary ─────────────────────────────
xgb_summary = features_df[features_df['xgb_flag'] == 1][
    ['address','xgb_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']
].sort_values(by='xgb_score', ascending=False).reset_index(drop=True)

xgb_summary.index += 1

print("\n── Isolation Forest + XGBoost Suspicious ──────────────────────")
print(xgb_summary.to_string())


XGBoost Suspicious: 5 / 5731

── Isolation Forest + XGBoost Suspicious ──────────────────────
                                      address  xgb_score   NI   NP      R_PT      R_MN      R_MM      R_Bal
1  0x28c6c06298d514db089934071355e5743bf21d60   0.944049   99   66  0.631153  0.166667  0.959269   0.900548
2  0x45300136662dd4e58fc0df61e6290dffd992b785   0.902340  561   22  0.500891  0.500000  0.930233 -41.156863
3  0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f   0.720773  132  110  0.692649  0.100000  0.670196   0.897104
4  0x1111111254eeb25477b68fb85ed929f73a960582   0.539253   44    0  0.141633  0.000000  0.000000   1.000000
5  0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2   0.539253  110    0  0.410125  0.000000  0.000000   1.000000


In [44]:

# ─────────────────────────────────────────────
# 1. XGBOOST SCORE DISTRIBUTION
# ─────────────────────────────────────────────
plt.figure(figsize=(8,5))
plt.hist(features_df['xgb_score'], bins=50)
plt.title("Anomaly Score Distribution For IF+XGBoost",fontsize=14,fontweight="bold")
plt.xlabel("Anomaly Score",fontsize=10,fontweight="bold")
plt.ylabel("Frequency",fontsize=10,fontweight="bold")
plt.xlim(-0.5,0.2)
plt.ylim(0,6000)
plt.savefig("Anomaly_Score_IF+XGBoost.png",dpi=600)
plt.close()

#print("\nSaved: xgb_score_distribution.png")


# ─────────────────────────────────────────────
# 2. TOP-K XGBOOST SUSPICIOUS ADDRESSES
# ─────────────────────────────────────────────
K = 7

top_k_xgb = features_df.sort_values(by='xgb_score', ascending=False).head(K)

print(f"\n── Top {K} XGBoost Suspicious Addresses ─────────────────────────")
print(top_k_xgb[['address','xgb_score','NI','NP','R_PT','R_MN','R_MM','R_Bal']].to_string(index=False))


# ─────────────────────────────────────────────
# 3. PCA VISUALIZATION (XGBOOST)
# ─────────────────────────────────────────────
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

features_df['pca1_xgb'] = X_pca[:,0]
features_df['pca2_xgb'] = X_pca[:,1]

plt.figure(figsize=(8,6))
plt.scatter(
    features_df['pca1_xgb'],
    features_df['pca2_xgb'],
    c=features_df['xgb_flag'],
)
plt.title("PCA Projection Of IF+XGBoost Detected Samples",fontsize=14,fontweight="bold")
plt.xlabel("PCA 1",fontsize=10,fontweight="bold")
plt.ylabel("PCA 2",fontsize=10,fontweight="bold")
plt.xlim(-0.8,0.8)
plt.ylim(-0.2,1.0)
plt.savefig("pca_IF+XGBoost.png",dpi=600)
plt.close()

#print("Saved: pca_xgb.png")


# ─────────────────────────────────────────────
# 4. SILHOUETTE SCORE (XGBOOST)
# ─────────────────────────────────────────────
labels_xgb = features_df['xgb_flag'].values

if len(np.unique(labels_xgb)) > 1:
    sil_score_xgb = silhouette_score(X_scaled, labels_xgb)
    print(f"\nXGBoost Silhouette Score: {sil_score_xgb:.4f}")
else:
    print("\nXGBoost Silhouette Score: Not applicable (only one class found)")


── Top 7 XGBoost Suspicious Addresses ─────────────────────────
                                   address  xgb_score  NI  NP     R_PT     R_MN     R_MM       R_Bal
0x28c6c06298d514db089934071355e5743bf21d60   0.944049  99  66 0.631153 0.166667 0.959269    0.900548
0x45300136662dd4e58fc0df61e6290dffd992b785   0.902340 561  22 0.500891 0.500000 0.930233  -41.156863
0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f   0.720773 132 110 0.692649 0.100000 0.670196    0.897104
0x1111111254eeb25477b68fb85ed929f73a960582   0.539253  44   0 0.141633 0.000000 0.000000    1.000000
0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2   0.539253 110   0 0.410125 0.000000 0.000000    1.000000
0x1ab4973a48dc892cd9971ece8e01dcc7688f8f23   0.457744  44  77 0.514392 0.142857 0.909091 -188.655237
0x4976a4a02f38326660d17bf34b431dc6e2eb2327   0.180572   0  77 0.500000 0.142857 0.549882    0.000000

XGBoost Silhouette Score: 0.4196


In [26]:
# ─────────────────────────────────────────────
# 2. CONTAMINATION RATIO
# ─────────────────────────────────────────────
suspicious = features_df['xgb_flag'].sum()
total = len(features_df)

print("\n─────────────────────────────────────────")
print("Contamination Ratio")
print("─────────────────────────────────────────")
print(f"Suspicious: {suspicious}")
print(f"Total: {total}")
print(f"Ratio: {suspicious/total:.5f}")


─────────────────────────────────────────
Contamination Ratio
─────────────────────────────────────────
Suspicious: 5
Total: 5731
Ratio: 0.00087


In [27]:
same_values = set(xgb_summary['address']).intersection(set(df4['data']))
print(same_values)

{'0x4e5b2e1dc63f6b91cb6cd759936495434c7e972f', '0x1111111254eeb25477b68fb85ed929f73a960582', '0x45300136662dd4e58fc0df61e6290dffd992b785', '0x28c6c06298d514db089934071355e5743bf21d60', '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2'}


In [28]:
results = []

# ─────────────────────────────────────────────
# 1. Isolation Forest
# ─────────────────────────────────────────────
labels_iso = features_df['is_suspicious'].values

if len(np.unique(labels_iso)) > 1:
    sil_iso = silhouette_score(X_scaled, labels_iso)
else:
    sil_iso = np.nan

ratio_iso = features_df['is_suspicious'].sum() / len(features_df)

results.append([
    "Isolation Forest",
    sil_iso,
    ratio_iso,
    features_df['is_suspicious'].sum()
])


# ─────────────────────────────────────────────
# 2. SMOTE
# ─────────────────────────────────────────────
labels_smote = features_df['smote_flag'].values

if len(np.unique(labels_smote)) > 1:
    sil_smote = silhouette_score(X_scaled, labels_smote)
else:
    sil_smote = np.nan

ratio_smote = features_df['smote_flag'].sum() / len(features_df)

results.append([
    "Isolation + SMOTE",
    sil_smote,
    ratio_smote,
    features_df['smote_flag'].sum()
])


# ─────────────────────────────────────────────
# 3. XGBoost
# ─────────────────────────────────────────────
labels_xgb = features_df['xgb_flag'].values

if len(np.unique(labels_xgb)) > 1:
    sil_xgb = silhouette_score(X_scaled, labels_xgb)
else:
    sil_xgb = np.nan

ratio_xgb = features_df['xgb_flag'].sum() / len(features_df)

results.append([
    "Isolation + XGBoost",
    sil_xgb,
    ratio_xgb,
    features_df['xgb_flag'].sum()
])


# ─────────────────────────────────────────────
# Create Comparison Table
# ─────────────────────────────────────────────
comparison_df = pd.DataFrame(
    results,
    columns=[
        "Method",
        "Silhouette Score",
        "Contamination Ratio",
        "Suspicious Count"
    ]
)

# Format
comparison_df["Silhouette Score"] = comparison_df["Silhouette Score"].round(4)
comparison_df["Contamination Ratio"] = comparison_df["Contamination Ratio"].round(5)

print("\n──────────── Model Comparison Table ────────────")
print(comparison_df.to_string(index=False))


──────────── Model Comparison Table ────────────
             Method  Silhouette Score  Contamination Ratio  Suspicious Count
   Isolation Forest            0.4414              0.00122                 7
  Isolation + SMOTE            0.5972              0.00017                 1
Isolation + XGBoost            0.4196              0.00087                 5
